# 类继承关系

```mermaid
classDiagram
    class MetaStatsBuilderMixin {
        <<metaclass>>
    }
    class MetaIndicatorBase {
        <<metaclass>>
    }
    class MetaPlotsBuilderMixin {
        <<metaclass>>
    }
    class Wrapping
    class StatsBuilderMixin
    class PlotsBuilderMixin
    class IndicatorBase
    class IndicatorFactory

    
    %% 继承关系
    MetaStatsBuilderMixin <|-- MetaIndicatorBase : metaclass
    MetaPlotsBuilderMixin <|-- MetaIndicatorBase : metaclass

    MetaStatsBuilderMixin <|-- StatsBuilderMixin : metaclass
    MetaPlotsBuilderMixin <|-- PlotsBuilderMixin : metaclass

    Wrapping <|-- IndicatorBase
    StatsBuilderMixin <|-- IndicatorBase
    MetaIndicatorBase <|-- IndicatorBase : metaclass
    PlotsBuilderMixin <|-- IndicatorBase
```

# prepare_params

预处理参数列表，进行类型转换、广播和形状调整。

参数：
- `param_list` (tp.Sequence[tp.Params])：待处理的参数列表
- `param_settings` (tp.KwargsLikeSequence)：参数设置，包含每个参数的配置选项
  - `is_tuple` (bool): 是否保持元组格式
  - `dtype` (tp.Optional[tp.DtypeLike]): 数据类型或映射
  - `is_array_like` (bool): 是否将数组视为单个参数
  - `bc_to_input` (bool): 是否广播到输入形状
  - `broadcast_kwargs` (tp.KwargsLike): 广播配置
- `input_shape` (tp.Optional[tp.Shape])：输入数据的形状，用于参数广播
- `to_2d` (bool)：是否将参数转换为二维形状

返回值：tp.List[tp.List]: 处理后的参数列表，每个参数都已经过预处理

## 例子

In [ ]:
import numpy as np
from vectorbt.indicators.factory import prepare_params

input_shape = (200, 4)

# 定义复杂的参数列表
param_list = [
    (10, 20, 30),                    # 元组参数
    [0.1, 0.2, 0.3, 0.4],           # 列表参数
    np.array([[1, 2], [3, 4]]),      # 二维数组
    'custom_method'                  # 字符串参数
]

# 定义复杂的参数设置
param_settings = [
    {'is_tuple': True, 'is_array_like': False},           # 保持元组格式
    {'bc_to_input': 1, 'is_tuple': False, 'is_array_like': False},  # 广播到列
    {'is_tuple': False, 'is_array_like': True},           # 数组视为单个参数
    {'is_tuple': False, 'is_array_like': False}           # 普通字符串
]

# 处理参数
result = prepare_params(param_list, param_settings, input_shape=input_shape)

print("输入形状:", input_shape)
print("原始参数:", param_list)
print("参数设置:", param_settings)
print("处理结果:")
for i, params in enumerate(result):
    print(f"  参数{i}: {params}")
    if hasattr(params[0], 'shape'):
        print(f"    形状: {[p.shape if hasattr(p, 'shape') else 'scalar' for p in params]}")
print()

In [ ]:
# 输入数据：股票价格数据 (1000行, 5列)
input_shape = (1000, 5)

# 参数：不同的移动平均窗口和类型
param_list = [
    [5, 10, 20, 50],           # 移动平均窗口
    ['sma', 'ema', 'wma']      # 移动平均类型
]

# 参数设置
param_settings = [
    {'bc_to_input': 1, 'is_tuple': False, 'is_array_like': False},  # 窗口参数广播到列
    {'is_tuple': False, 'is_array_like': False}                     # 类型参数
]

# 处理参数
processed_params = prepare_params(param_list, param_settings, input_shape=input_shape)

print("移动平均线参数处理:")
print("原始参数:", param_list)
print("处理后参数:", processed_params)

## 源码

```python
def prepare_params(param_list: tp.Sequence[tp.Params],
                   param_settings: tp.KwargsLikeSequence = None,
                   input_shape: tp.Optional[tp.Shape] = None,
                   to_2d: bool = False) -> tp.List[tp.List]:

    new_param_list = []
    for i, params in enumerate(param_list):
        _param_settings = resolve_dict(param_settings, i=i)
        is_tuple = _param_settings.get('is_tuple', False)
        dtype = _param_settings.get('dtype', None)
        if checks.is_mapping_like(dtype):
            if checks.is_namedtuple(dtype):
                params = map_enum_fields(params, dtype)
            else:
                params = apply_mapping(params, dtype)
        is_array_like = _param_settings.get('is_array_like', False)
        bc_to_input = _param_settings.get('bc_to_input', False)
        broadcast_kwargs = _param_settings.get('broadcast_kwargs', dict(require_kwargs=dict(requirements='W')))

        new_params = params_to_list(params, is_tuple, is_array_like)
        if bc_to_input is not False:
            # Broadcast to input or its axis
            if is_tuple:
                raise ValueError("Cannot broadcast to input if tuple")
            if input_shape is None:
                raise ValueError("Cannot broadcast to input if input shape is unknown. Pass input_shape.")
            if bc_to_input is True:
                to_shape = input_shape
            else:
                checks.assert_in(bc_to_input, (0, 1))
                # Note that input_shape can be 1D
                if bc_to_input == 0:
                    to_shape = (input_shape[0],)
                else:
                    to_shape = (input_shape[1],) if len(input_shape) > 1 else (1,)
            _new_params = reshape_fns.broadcast(
                *new_params,
                to_shape=to_shape,
                **broadcast_kwargs
            )
            if len(new_params) == 1:
                _new_params = [_new_params]
            else:
                _new_params = list(_new_params)
            if to_2d and bc_to_input is True:
                # If inputs are meant to reshape to 2D, do the same to parameters
                # But only to those that fully resemble inputs (= not raw)
                __new_params = _new_params.copy()
                for j, param in enumerate(__new_params):
                    keep_raw = broadcast_kwargs.get('keep_raw', False)
                    if keep_raw is False or (isinstance(keep_raw, (tuple, list)) and not keep_raw[j]):
                        __new_params[j] = reshape_fns.to_2d(param)
                new_params = __new_params
            else:
                new_params = _new_params
        new_param_list.append(new_params)
    return new_param_list
```

# build_columns
构建多层级列索引，将参数值作为新的列级别叠加在输入列之上。

参数：
- `param_list` (tp.Sequence[tp.Sequence[tp.Param]])：参数列表，每个参数包含多个值
- `input_columns` (tp.IndexLike)：输入数据的列索引
- `level_names` (tp.Optional[tp.Sequence[str]])：参数的名称列表
- `hide_levels` (tp.Optional[tp.Sequence[int]])：需要隐藏的级别索引列表
- `param_settings` (tp.KwargsLikeSequence)：参数设置，包含每个参数的配置选项
- `per_column (bool)`：是否按列处理参数
- `ignore_default (bool)`：是否忽略默认值
- `**kwargs`：传递给索引堆叠函数的额外参数

返回值：
    tp.Tuple[tp.List[tp.Index], tp.Index]: 参数索引列表和新的列索引

## 例子

In [ ]:
import numpy as np
import pandas as pd
from vectorbt.indicators.factory import build_columns

"""基本的多层级列索引构建示例"""
print("=== 示例1：基本的多层级列索引构建 ===")

# 定义参数列表
param_list = [
    [10, 20, 50],           # 第一个参数：窗口大小
    [0.1, 0.2, 0.3]        # 第二个参数：平滑系数
]

# 定义输入列
input_columns = pd.Index(['AAPL', 'GOOGL', 'MSFT'])

# 定义级别名称
level_names = ['window', 'alpha']

# 构建多层级列索引
param_indexes, new_columns = build_columns(
    param_list=param_list,
    input_columns=input_columns,
    level_names=level_names
)

print("原始输入列:", input_columns.tolist())
print("参数列表:", param_list)
print("级别名称:", level_names)
print("\n参数索引:")
for i, idx in enumerate(param_indexes):
    print(f"  参数{i} ({level_names[i]}): {idx.tolist()}")

print("\n新的多层级列索引:")
print(new_columns)

In [ ]:
print("=== 示例2：隐藏特定级别 ===")

# 定义参数列表 - 确保所有参数列表长度相同
param_list = [
    [5, 10, 20],            # 窗口大小
    ['sma', 'ema', 'wma'],  # 移动平均类型 - 修改为3个值
    [0.1, 0.2, 0.3]        # 平滑系数 - 修改为3个值
]

# 定义输入列
input_columns = pd.Index(['BTC', 'ETH'])

# 定义级别名称
level_names = ['window', 'method', 'alpha']

# 隐藏第二个级别（method）
hide_levels = [1]

# 构建多层级列索引
param_indexes, new_columns = build_columns(
    param_list=param_list,
    input_columns=input_columns,
    level_names=level_names,
    hide_levels=hide_levels
)

print("原始输入列:", input_columns.tolist())
print("参数列表:", param_list)
print("级别名称:", level_names)
print("隐藏级别:", hide_levels)

print("\n新的多层级列索引（隐藏method级别）:")
print(new_columns)

In [ ]:
print("=== 示例6：复杂参数组合 ===")

# 定义复杂的参数列表 - 确保所有参数列表长度相同
param_list = [
    [10, 20, 50],                    # 窗口大小
    ['RSI', 'MACD', 'BB'],           # 指标类型
    [0.1, 0.2, 0.3],                # 平滑系数 - 修改为3个值
    ['buy', 'sell', 'hold']          # 信号类型 - 修改为3个值
]

# 定义输入列
input_columns = pd.Index(['Crypto_A', 'Crypto_B'])

# 定义级别名称
level_names = ['window', 'indicator', 'smooth', 'signal']

# 隐藏某些级别
hide_levels = [3]  # 隐藏信号类型级别

# 构建多层级列索引
param_indexes, new_columns = build_columns(
    param_list=param_list,
    input_columns=input_columns,
    level_names=level_names,
    hide_levels=hide_levels
)

print("原始输入列:", input_columns.tolist())
print("参数列表:")
for i, params in enumerate(param_list):
    print(f"  {level_names[i]}: {params}")
print("隐藏级别:", hide_levels)

print("\n新的多层级列索引:")
print(new_columns)

## 源码

```python
def build_columns(param_list: tp.Sequence[tp.Sequence[tp.Param]],
                  input_columns: tp.IndexLike,
                  level_names: tp.Optional[tp.Sequence[str]] = None,
                  hide_levels: tp.Optional[tp.Sequence[int]] = None,
                  param_settings: tp.KwargsLikeSequence = None,
                  per_column: bool = False,
                  ignore_default: bool = False,
                  **kwargs) -> tp.Tuple[tp.List[tp.Index], tp.Index]:

    if level_names is not None:
        checks.assert_len_equal(param_list, level_names)
    if hide_levels is None:
        hide_levels = []
    input_columns = index_fns.to_any_index(input_columns)

    param_indexes = []
    shown_param_indexes = []
    for i in range(len(param_list)):
        params = param_list[i]
        level_name = None
        if level_names is not None:
            level_name = level_names[i]
        if per_column:
            param_index = index_fns.index_from_values(params, name=level_name)
        else:
            _param_settings = resolve_dict(param_settings, i=i)
            _per_column = _param_settings.get('per_column', False)
            if _per_column:
                param_index = None
                for param in params:
                    bc_param = np.broadcast_to(param, (len(input_columns),))
                    _param_index = index_fns.index_from_values(bc_param, name=level_name)
                    if param_index is None:
                        param_index = _param_index
                    else:
                        param_index = param_index.append(_param_index)
                if len(param_index) == 1 and len(input_columns) > 1:
                    # When using flexible column-wise parameters
                    param_index = index_fns.repeat_index(
                        param_index,
                        len(input_columns),
                        ignore_default=ignore_default
                    )
            else:
                param_index = index_fns.index_from_values(param_list[i], name=level_name)
                param_index = index_fns.repeat_index(
                    param_index,
                    len(input_columns),
                    ignore_default=ignore_default
                )
        param_indexes.append(param_index)
        if i not in hide_levels:
            shown_param_indexes.append(param_index)
    if len(shown_param_indexes) > 0:
        if not per_column:
            n_param_values = len(param_list[0]) if len(param_list) > 0 else 1
            input_columns = index_fns.tile_index(
                input_columns,
                n_param_values,
                ignore_default=ignore_default
            )
        stacked_columns = index_fns.stack_indexes([*shown_param_indexes, input_columns], **kwargs)
        return param_indexes, stacked_columns
    return param_indexes, input_columns
```

# run_pipeline
用于标准化指标计算流程：
1. 输入数据预处理和广播
2. 参数处理和组合生成
3. 执行自定义计算函数
4. 输出结果后处理
5. 构建最终的数据结构

```python
def run_pipeline(
        num_ret_outputs: int, # 自定义函数返回的输出数组数量
        custom_func: tp.Callable, # 自定义计算函数
        *args, # 传递给自定义函数的位置参数
        require_input_shape: bool = False, # 是否需要输入形状，如果为True则会传递input_shape并检查
        input_shape: tp.Optional[tp.RelaxedShape] = None, # 输入数据的广播目标形状
        input_index: tp.Optional[tp.IndexLike] = None, # 输入数据的索引
        input_columns: tp.Optional[tp.IndexLike] = None, # 输入数据的列索引
        input_list: tp.Optional[tp.Sequence[tp.ArrayLike]] = None, # 输入数组列表
        in_output_list: tp.Optional[tp.Sequence[tp.ArrayLike]] = None, # 就地输出数组列表
        in_output_settings: tp.KwargsLikeSequence = None, # 就地输出的设置
        broadcast_kwargs: tp.KwargsLike = None, # 广播函数的关键字参数
        param_list: tp.Optional[tp.Sequence[tp.Param]] = None, # 参数列表
        param_product: bool = False, # 是否构建参数的笛卡尔积
        param_settings: tp.KwargsLikeSequence = None, # 参数设置
        run_unique: bool = False, # 是否只运行唯一的参数组合
        silence_warnings: bool = False, # 是否隐藏警告
        per_column: bool = False, # 是否按列处理参数
        pass_col: bool = False, # 是否传递列索引到自定义函数
        keep_pd: bool = False, # 是否保持pandas对象格式
        to_2d: bool = True, # 是否将输入重塑为二维数组
        as_lists: bool = False, # 是否以列表形式传递输入和参数
        pass_input_shape: bool = False, # 是否传递输入形状到自定义函数
        pass_flex_2d: bool = False, # 是否传递flex_2d参数到自定义函数
        level_names: tp.Optional[tp.Sequence[str]] = None, # 参数级别名称列表
        hide_levels: tp.Optional[tp.Sequence[int]] = None, # 需要隐藏的级别索引
        stacking_kwargs: tp.KwargsLike = None, # 索引堆叠的关键字参数
        return_raw: bool = False, # 是否返回原始输出
        use_raw: tp.Optional[RawOutputT] = None, # 使用原始结果而不是运行计算
        wrapper_kwargs: tp.KwargsLike = None, # 数组包装器的关键字参数
        seed: tp.Optional[int] = None, # 随机种子
        **kwargs) -> tp.Union[CacheOutputT, RawOutputT, PipelineOutputT]
```

## 例子

In [ ]:
import numpy as np
import pandas as pd
import vectorbt as vbt
from vectorbt.indicators.factory import run_pipeline
from vectorbt.base.array_wrapper import ArrayWrapper

### 基本的移动平均线计算

In [ ]:
"""基本的移动平均线计算示例"""
print("=== 示例1：基本的移动平均线计算 ===")

# 创建模拟价格数据
np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=100, freq='D')
prices = np.random.randn(100, 3).cumsum(axis=0) + 100  # 3只股票的价格数据

# 创建DataFrame
df = pd.DataFrame(prices, index=dates, columns=['AAPL', 'GOOGL', 'MSFT'])

# 修正的移动平均线计算函数
def simple_ma(close, window_list):
    """简单的移动平均线计算 - 修正版本"""
    # window_list 是一个列表，包含所有窗口值
    results = []
    for window in window_list:
        result = np.full_like(close, np.nan)
        for i in range(window - 1, len(close)):
            result[i] = np.mean(close[i - window + 1:i + 1])
        results.append(result)
    return np.column_stack(results)

# 使用run_pipeline计算移动平均线
result = run_pipeline(
    num_ret_outputs=1,  # 返回1个输出数组
    custom_func=simple_ma,  # 自定义计算函数
    input_list=[df.values],  # 输入数据列表
    param_list=[[5, 10, 20]],  # 参数列表：不同的窗口大小
    input_index=df.index,  # 输入索引
    input_columns=df.columns,  # 输入列
    level_names=['window'],  # 参数级别名称
    to_2d=True  # 转换为二维数组
)

# 解包结果
wrapper, input_list, input_mapper, in_output_list, output_list, param_list, mapper_list, other_list = result

print("输入数据形状:", df.shape)
print("参数列表:", param_list)
print("输出数据形状:", output_list[0].shape)

# # 创建结果DataFrame
# result_df = pd.DataFrame(output_list[0], index=df.index, columns=wrapper.columns)
# print("\n结果DataFrame:")
# print(result_df.head(30))
# print("\n列索引结构:")
# print(result_df.columns)

print("wrapper.columns:")
print(wrapper.columns)
print("wrapper.index:")
print(wrapper.index)
print("input_list:")
print(input_list)
print("input_mapper:")
print(input_mapper)
print("in_output_list:")
print(in_output_list)
print("output_list:")
print(output_list)
print("param_list:")
print(param_list)
print("mapper_list:")
print(mapper_list)
print("other_list:")
print(other_list)

### RSI计算

In [ ]:
print("\n=== 示例2：RSI计算 ===")

# 创建模拟价格数据
np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=200, freq='D')
prices = np.random.randn(200, 2).cumsum(axis=0) + 100

df = pd.DataFrame(prices, index=dates, columns=['BTC', 'ETH'])

# 修正的RSI计算函数
def calculate_rsi(close, window_list, smooth_list):
    """计算RSI指标 - 修正版本"""
    def ema(data, period):
        alpha = 2.0 / (period + 1)
        ema_values = np.full_like(data, np.nan)
        ema_values[0] = data[0]
        for i in range(1, len(data)):
            ema_values[i] = alpha * data[i] + (1 - alpha) * ema_values[i-1]
        return ema_values
    
    results = []
    for window, smooth in zip(window_list, smooth_list):
        # 确保close是一维数组
        if close.ndim > 1:
            close_1d = close.flatten()
        else:
            close_1d = close
            
        delta = np.diff(close_1d, prepend=close_1d[0])
        gain = np.where(delta > 0, delta, 0)
        loss = np.where(delta < 0, -delta, 0)
        
        avg_gain = np.full_like(gain, np.nan)
        avg_loss = np.full_like(loss, np.nan)
        
        avg_gain[window-1] = np.mean(gain[:window])
        avg_loss[window-1] = np.mean(loss[:window])
        
        for i in range(window, len(gain)):
            avg_gain[i] = smooth * gain[i] + (1 - smooth) * avg_gain[i-1]
            avg_loss[i] = smooth * loss[i] + (1 - smooth) * avg_loss[i-1]
        
        rs = avg_gain / avg_loss
        rsi = 100 - (100 / (1 + rs))
        
        # 恢复原始形状
        if close.ndim > 1:
            rsi = rsi.reshape(close.shape)
        results.append(rsi)
    
    return np.column_stack(results)

# 使用run_pipeline计算RSI（不使用param_product）
result = run_pipeline(
    num_ret_outputs=1,
    custom_func=calculate_rsi,
    input_list=[df.values],
    param_list=[[14, 21], [0.1, 0.2]],  # 窗口大小和平滑因子
    input_index=df.index,
    input_columns=df.columns,
    level_names=['window', 'smooth'],
    param_product=False,  # 不使用参数笛卡尔积
    to_2d=True
)

# 解包结果
wrapper, input_list, input_mapper, in_output_list, output_list, param_list, mapper_list, other_list = result

print("输入数据形状:", df.shape)
print("参数组合数量:", len(param_list[0]))
print("输出数据形状:", output_list[0].shape)

print("wrapper.columns:")
print(wrapper.columns)
print("wrapper.index:")
print(wrapper.index)
print("input_list:")
print(input_list)
print("input_mapper:")
print(input_mapper)
print("in_output_list:")
print(in_output_list)
print("output_list:")
print(output_list)
print("param_list:")
print(param_list)
print("mapper_list:")
print(mapper_list)
print("other_list:")
print(other_list)

print(pd.DataFrame(output_list[0], index=df.index, columns=wrapper.columns))

# combine_objs
组合/比较对象以生成信号或其他结果。

参数
- `obj` (tp.SeriesFrame)：主要的数据对象（Series或DataFrame）
- `other` (tp.MaybeTupleList[tp.Union[tp.ArrayLike, BaseAccessor]])：要比较的对象或对象列表
- `*args`: 传递给组合函数的位置参数
- `level_name` (tp.Optional[str])：当处理多个对象时，新创建的列级别名称
- `keys` (tp.Optional[tp.IndexLike])：用于标识不同对象的键
- `allow_multiple` (bool)：是否允许多个对象的比较
- `**kwargs`：传递给组合函数的关键字参数

返回值：tp.SeriesFrame: 组合后的结果，通常是布尔值表示的信号

```python
def combine_objs(obj: tp.SeriesFrame,
                 other: tp.MaybeTupleList[tp.Union[tp.ArrayLike, BaseAccessor]],
                 *args, level_name: tp.Optional[str] = None,
                 keys: tp.Optional[tp.IndexLike] = None,
                 allow_multiple: bool = True,
                 **kwargs) -> tp.SeriesFrame:

    if allow_multiple and isinstance(other, (tuple, list)):
        if keys is None:
            keys = index_fns.index_from_values(other, name=level_name)
    return obj.vbt.combine(other, *args, keys=keys, concat=True, allow_multiple=allow_multiple, **kwargs)
```

## 例子

In [ ]:
import numpy as np
import pandas as pd
import vectorbt as vbt
from vectorbt.indicators.factory import combine_objs

# 设置随机种子以确保结果可重现
np.random.seed(42)

# 1. 创建模拟价格数据
print("1. 创建模拟价格数据")
dates = pd.date_range('2024-01-01', periods=100, freq='D')
prices = np.random.randn(100, 3).cumsum(axis=0) + 100
df = pd.DataFrame(prices, index=dates, columns=['AAPL', 'GOOGL', 'MSFT'])
print(f"价格数据形状: {df.shape}")
print(f"价格数据前5行:\n{df.head()}\n")

# 2. 计算移动平均线
print("2. 计算移动平均线")
ma_short = vbt.MA.run(df, window=10).ma
ma_long = vbt.MA.run(df, window=30).ma
print(f"短期MA形状: {ma_short.shape}")
print(f"长期MA形状: {ma_long.shape}")
print(f"短期MA前5行:\n{ma_short.head()}\n")

# 3. 基本用法：单个对象比较
print("3. 基本用法：单个对象比较")
print("3.1 价格与移动平均线交叉信号")

# 价格上穿短期移动平均线 - 使用正确的combine_func
def crossed_above_func(x, y):
    """自定义上穿函数"""
    return x > y

price_cross_short = combine_objs(
    df, 
    ma_short, 
    combine_func=crossed_above_func
)
print(f"价格上穿短期MA信号形状: {price_cross_short.shape}")
print(f"价格上穿短期MA信号前30行:\n{price_cross_short.head(30)}\n")

# 价格下穿短期移动平均线
def crossed_below_func(x, y):
    """自定义下穿函数"""
    return x < y

price_cross_short_below = combine_objs(
    df, 
    ma_short, 
    combine_func=crossed_below_func
)
print(f"价格下穿短期MA信号前30行:\n{price_cross_short_below.head(30)}\n")

# class MetaIndicatorBase(type(StatsBuilderMixin), type(PlotsBuilderMixin))
指标基础元类。

```python
class MetaIndicatorBase(type(StatsBuilderMixin), type(PlotsBuilderMixin)):
    pass
```

# class IndicatorBase(Wrapping, StatsBuilderMixin, PlotsBuilderMixin, metaclass=MetaIndicatorBase)
基础指标类，所有技术指标的基础类。提供：
- 数据包装功能（来自 `Wrapping`）
- 统计计算功能（来自 `StatsBuilderMixin`）
- 绘图功能（来自 `PlotsBuilderMixin`）
- 统一的指标接口

```python
class IndicatorBase(Wrapping, StatsBuilderMixin, PlotsBuilderMixin, metaclass=MetaIndicatorBase):
    _short_name: 指标的简短名称
    _level_names: 参数级别名称
    _input_names: 输入数据名称
    _param_names: 参数名称
    _in_output_names: 就地输出名称
    _output_names: 输出名称
    _output_flags: 输出标志
```

### `__init__`

```python
def __init__(self,
                wrapper: ArrayWrapper,
                input_list: InputListT,
                input_mapper: InputMapperT,
                in_output_list: InOutputListT,
                output_list: OutputListT,
                param_list: ParamListT,
                mapper_list: MapperListT,
                short_name: str,
                level_names: tp.Tuple[str, ...]) -> None:
    Wrapping.__init__(
        self,
        wrapper,
        input_list=input_list,
        input_mapper=input_mapper,
        in_output_list=in_output_list,
        output_list=output_list,
        param_list=param_list,
        mapper_list=mapper_list,
        short_name=short_name,
        level_names=level_names
    )
    StatsBuilderMixin.__init__(self)
    PlotsBuilderMixin.__init__(self)

    if input_mapper is not None:
        checks.assert_equal(input_mapper.shape[0], wrapper.shape_2d[1])
    for ts in input_list:
        checks.assert_equal(ts.shape[0], wrapper.shape_2d[0])
    for ts in in_output_list + output_list:
        checks.assert_equal(ts.shape, wrapper.shape_2d)
    for params in param_list:
        checks.assert_len_equal(param_list[0], params)
    for mapper in mapper_list:
        checks.assert_equal(len(mapper), wrapper.shape_2d[1])
    checks.assert_instance_of(short_name, str)
    checks.assert_len_equal(level_names, param_list)

    setattr(self, '_short_name', short_name)
    setattr(self, '_level_names', level_names)

    for i, ts_name in enumerate(self.input_names):
        setattr(self, f'_{ts_name}', input_list[i])
    setattr(self, '_input_mapper', input_mapper)
    for i, in_output_name in enumerate(self.in_output_names):
        setattr(self, f'_{in_output_name}', in_output_list[i])
    for i, output_name in enumerate(self.output_names):
        setattr(self, f'_{output_name}', output_list[i])
    for i, param_name in enumerate(self.param_names):
        setattr(self, f'_{param_name}_list', param_list[i])
        setattr(self, f'_{param_name}_mapper', mapper_list[i])
    if len(self.param_names) > 1:
        tuple_mapper = list(zip(*list(mapper_list)))
        setattr(self, '_tuple_mapper', tuple_mapper)
```

## run

```python
@classmethod
def _run(cls: tp.Type[IndicatorBaseT], *args, **kwargs) -> RunOutputT:
    raise NotImplementedError

@classmethod
def run(cls: tp.Type[IndicatorBaseT], *args, **kwargs) -> RunOutputT:
    return cls._run(*args, **kwargs)
```

## run_combs

```python
@classmethod
def _run_combs(cls: tp.Type[IndicatorBaseT], *args, **kwargs) -> RunCombsOutputT:
    raise NotImplementedError

@classmethod
def run_combs(cls: tp.Type[IndicatorBaseT], *args, **kwargs) -> RunCombsOutputT:
    return cls._run_combs(*args, **kwargs)
```

# class IndicatorFactory 

## `__init__`

```python
def __init__(self,
            class_name: str = 'Indicator', # 创建的指标类名称，默认为'Indicator'
            class_docstring: str = '', # 创建的指标类的文档字符串
            module_name: tp.Optional[str] = __name__, # 指定类所属的模块名称
            short_name: tp.Optional[str] = None, # 指标的简短名称
            prepend_name: bool = True, # 是否在每个参数级别前添加short_name
            input_names: tp.Optional[tp.Sequence[str]] = None, # 输入数组名称列表
            param_names: tp.Optional[tp.Sequence[str]] = None, # 参数名称列表
            in_output_names: tp.Optional[tp.Sequence[str]] = None, # 就地输出数组名称列表
            output_names: tp.Optional[tp.Sequence[str]] = None, # 输出数组名称列表
            output_flags: tp.KwargsLike = None, # 就地输出和常规输出标志字典
            custom_output_props: tp.KwargsLike = None, # 用户自定义函数字典
            attr_settings: tp.KwargsLike = None, # 按属性名称的设置字典
            metrics: tp.Optional[tp.Kwargs] = None, # 统计构建器支持的指标
            stats_defaults: tp.Union[None, tp.Callable, tp.Kwargs] = None, # 统计默认值
            subplots: tp.Optional[tp.Kwargs] = None, # 子图配置
            plots_defaults: tp.Union[None, tp.Callable, tp.Kwargs] = None) -> None: # 绘图默认值
```

## from_custom_func
基于自定义计算函数构建指标类 `IndicatorBase`。
```python
def from_custom_func(self, # 基于自定义计算函数构建指标类
                    custom_func: tp.Callable, # 自定义计算函数
                    require_input_shape: bool = False, # 是否需要输入形状
                    param_settings: tp.KwargsLike = None, # 参数设置字典（按名称索引）
                    in_output_settings: tp.KwargsLike = None, # 就地输出设置字典（按名称索引）
                    hide_params: tp.Optional[tp.Sequence[str]] = None, # 要隐藏列级别的参数名称列表
                    hide_default: bool = True, # 是否隐藏具有默认值的参数的列级别
                    var_args: bool = False, # 运行方法是否接受可变参数(*args)
                    keyword_only_args: bool = False, # 运行方法是否接受仅关键字参数
                    **pipeline_kwargs)  # 传递给run_pipeline的关键字参数
                    -> tp.Type[IndicatorBase]:
```

### 例子

## from_apply_func
```python
def from_apply_func(self,
                    apply_func: tp.Callable, # 应用函数
                    cache_func: tp.Optional[tp.Callable] = None, # 缓存函数
                    pass_packed: bool = False, # 是否为输入、就地输出和参数传递打包的元组
                    kwargs_to_args: tp.Optional[tp.Sequence[str]] = None, # 从kwargs字典中作为位置参数传递的关键字参数列表
                    numba_loop: bool = False, # 是否使用Numba循环
                    **kwargs) # 传递给IndicatorFactory.from_custom_func的关键字参数
                    -> tp.Type[IndicatorBase]: 
```